# Classical vs. neural methods: DoF, error and computation time

In the previous tutorials we approximated the solutions of two problems with
physics-informed neural networks (PINNs): a parametric 2D Laplacian
(stationary, with a parameter $\mu$) and a 2D rotating transport (advection)
equation (time-dependent).

Here we put those neural methods side by side with the classical ones, the
finite element method (FEM) and the discontinuous Galerkin method (DG), and
compare them on the number of degrees of freedom (DoF) of the approximation,
the error of the computed solution and the computation time required to
obtain it.

The two families of methods are not directly comparable. Classical methods
solve a *linear system* whose size is fixed by the mesh, while neural methods
*optimize* the weights of a network, so their cost is governed by the number
of training epochs and collocation points, not by the mesh. The comparison is
therefore most meaningful when we fix a target accuracy and ask how much work
each method needs to reach it.

## 1/ A parametric 2D Laplacian on the disk: FEM, DG and PINN

We consider the equation

$$- \mu \Delta u = 4\mu^2\, x\,(2-r^2)\, e^{-r^2}, \qquad r^2 = x^2 + y^2$$

on the unit disk $\Omega := \{(x,y) : x^2 + y^2 \le 1\}$ with parameter
$\mu\in \mathbb{M}=[1,2]$, subject to the Dirichlet condition $u = 0$ on
$\partial\Omega$. The exact solution is

$$ u(x,y,\mu) = \mu \left(e^{-r^2} - e^{-1}\right) x, $$

which vanishes on $\partial\Omega$, is not a polynomial, and is not radial
(the factor $x = r\cos\theta$ breaks the radial symmetry).

We solve this problem *in a monolithic fashion*, without snapshots on $\mu$,
on the whole continuous interval $\mu \in \mathbb{M}=[1,2]$.


### Imports

We import the modules for the classical methods (mesh, basis, FEM/DG schemes,
weak forms, error analysis) and for the neural methods (domains, samplers,
networks, approximation spaces, the PINN projector).


In [ ]:
# # uncomment the line below if you are on Google Colab
# !pip install scimba

import timeit
from pathlib import Path

import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np

# --- classical (FEM / DG) ---
from scimba_jax.linear_approximation.basis.analytic_bases import (
    local_lagrange_basis,
    local_lagrange_basis_by_logical,
    local_taylor_basis,
    local_taylor_basis_by_logical,
)
from scimba_jax.linear_approximation.basis.dof_map import UnstructuredNodalDofMap
from scimba_jax.linear_approximation.basis.general_bases import AnalyticBasis
from scimba_jax.linear_approximation.error_analysis import l2_error
from scimba_jax.linear_approximation.galerkin.dg.elliptic_dg_scheme import (
    EllipticDGscheme,
)
from scimba_jax.linear_approximation.galerkin.dg.flux import SIPGFlux
from scimba_jax.linear_approximation.galerkin.fem.elliptic_fe_scheme import (
    EllipticFEscheme,
)
from scimba_jax.linear_approximation.meshes.unstructured_mesh import UnstructuredMesh
from scimba_jax.linear_approximation.quad.gauss_quad import UnitSquareTensorized
from scimba_jax.linear_approximation.variables.variables_dg import VariablesDG
from scimba_jax.linear_approximation.variables.variables_fe import VariablesFE
from scimba_jax.physical_models.abstract_physical_weak_model import (
    AbstractPhysicalWeakModel,
)
from scimba_jax.physical_models.classical_weakform.laplacian_weak_form import (
    LaplacianWeakForm,
)

# --- neural (PINN) ---
from scimba_jax.domains.meshless_domains.domains_2d import Disk2D
from scimba_jax.nonlinear_approximation.approximation_spaces.approximation_spaces import (
    ApproximationSpace,
)
from scimba_jax.nonlinear_approximation.integration.monte_carlo import (
    DomainSampler,
    TensorizedSampler,
)
from scimba_jax.nonlinear_approximation.integration.monte_carlo_parameters import (
    UniformParametricSampler,
)
from scimba_jax.nonlinear_approximation.networks.mlp import MLP
from scimba_jax.nonlinear_approximation.numerical_solvers.projectors import Projector
from scimba_jax.physical_models.elliptic_pde.laplacians import (
    ParametricLaplacianDirichletND,
)

## Student version: a notebook with holes

This notebook is the **student version** of `solutions/3_comparison.ipynb`: some pieces of
code have been removed and replaced by calls to `TODO("...")`.

Running a cell that still contains a `TODO(...)` raises a `NotImplementedError`
with a hint. Replace each `TODO(...)` with your own code -- the surrounding text
usually contains the answer -- then re-run the cell.

The complete, filled notebook is `solutions/3_comparison.ipynb`.


In [ ]:
def TODO(hint: str = ""):
    """Placeholder for a hole to fill in: raises a clear error when executed."""
    raise NotImplementedError("hole to fill" + (f" -- {hint}" if hint else ""))

### The problem data

The right-hand side and the exact solution, as functions of $(x,y)$ and $\mu$.


In [ ]:
def f_rhs(xy: jnp.ndarray, mu: jnp.ndarray) -> jnp.ndarray:
    """Right-hand side of the PDE: f = 4 mu^2 x (2 - r^2) exp(-r^2)."""
    x, r2 = xy[..., 0:1], xy[..., 0:1] ** 2 + xy[..., 1:2] ** 2
    return 4 * mu[..., 0:1] ** 2 * x * (2.0 - r2) * jnp.exp(-r2)


def analytic_solution(xy: jnp.ndarray, mu: jnp.ndarray) -> jnp.ndarray:
    """Exact solution: u = mu (exp(-r^2) - exp(-1)) x, zero at r = 1, non-radial."""
    x, r2 = xy[..., 0:1], xy[..., 0:1] ** 2 + xy[..., 1:2] ** 2
    return mu * (jnp.exp(-r2) - jnp.exp(-1.0)) * x

### The classical methods: FEM and DG

Both classical methods are built on the same ingredients: a mesh of the
domain, a basis (the shape functions), and the variables that hold the
degrees of freedom. The only difference between FEM and DG is how the basis is
glued across cell interfaces. In FEM, the basis is *continuous* across
interfaces (Lagrange basis), so the degrees of freedom are shared by
neighbouring cells. In DG, the basis is *discontinuous* (Taylor basis), so
each cell carries its own degrees of freedom, and the coupling between cells
is enforced through a *numerical flux* (here the symmetric interior penalty
Galerkin, SIPG, flux).

We build a factory that, given a mesh resolution and a polynomial order,
returns a *solved* FEM or DG assembler for the Laplacian. The disk meshes
are pre-generated with gmsh (see ``generate_meshes.py``) and stored as
``.npz`` files in the ``meshes/`` folder: ``n_cells`` is the number of
cells per diameter of the disk.


In [ ]:
PHYSICAL_DIM = 2
QUAD_ORDER = 4
OUT_DIM = 1
POLY_ORDER = TODO("polynomial order of the FEM/DG basis (a first-order, i.e. linear, basis is a sensible default)")

def find_mesh_dir() -> Path:
    """Locate the folder holding the pre-generated disk meshes.

    Walks up from the current working directory, so the notebooks run both
    from ``tp_scimba/`` and from ``tp_scimba/solutions/``.
    """
    for base in (Path.cwd(), *Path.cwd().parents):
        for cand in (base / "meshes", base / "tp_scimba" / "meshes"):
            if cand.is_dir():
                return cand
    raise FileNotFoundError(
        "could not locate the 'meshes' folder; run generate_meshes.py first"
    )


def load_disk_mesh(n_cells):
    """Load the pre-generated gmsh mesh of the unit disk with n_cells per diameter."""
    data = np.load(find_mesh_dir() / f"disk_{n_cells}.npz")
    return UnstructuredMesh(
        nodes=data["nodes"],
        cells=data["cells"],
        ref_quad=UnitSquareTensorized(dim=PHYSICAL_DIM, order=QUAD_ORDER),
        order=int(data["geom_order"]),
    )


def lagrange(coords, i, mesh):
    return local_lagrange_basis(coords, i, mesh, order=POLY_ORDER, out_dim=OUT_DIM)


def lagrange_by_logical(y, i, mesh):
    return local_lagrange_basis_by_logical(y, i, mesh, order=POLY_ORDER, out_dim=OUT_DIM)


def taylor(coords, i, mesh):
    return local_taylor_basis(coords, i, mesh, order=POLY_ORDER, out_dim=OUT_DIM)


def taylor_by_logical(y, i, mesh):
    return local_taylor_basis_by_logical(y, i, mesh, order=POLY_ORDER, out_dim=OUT_DIM)


def zero_bc(__x):
    return jnp.zeros(OUT_DIM)


def u_exact(x):
    """Exact solution at a fixed parameter value (mu = 1.5, the middle of M)."""
    mu = 1.5
    r2 = x[0] ** 2 + x[1] ** 2
    return (mu * (jnp.exp(-r2) - jnp.exp(-1.0)) * x[0]).reshape(OUT_DIM)


def f_rhs_classical(x):
    """Right-hand side at a fixed parameter value (mu = 1.5).

    The classical model solves -Δu = f (LaplacianWeakForm), while the PDE is
    -mu Δu = f_rhs; at fixed mu the right-hand side is therefore f_rhs / mu.
    """
    mu = 1.5
    r2 = x[0] ** 2 + x[1] ** 2
    return 4 * mu * x[0] * (2.0 - r2) * jnp.exp(-r2)


def make_fem(n_cells, poly_order):
    """Build and solve a FEM assembler for the Laplacian on the disk."""
    global POLY_ORDER
    POLY_ORDER = poly_order
    mesh = load_disk_mesh(n_cells)
    basis = AnalyticBasis(
        nb_basis=(poly_order + 1) ** PHYSICAL_DIM,
        out_dim=OUT_DIM,
        mesh=mesh,
        local_basis=lagrange,
        local_basis_by_logical=lagrange_by_logical,
        basis_type="scalar",
    )
    variables = VariablesFE(
        basis=basis, nb_variables=OUT_DIM, dof_map=UnstructuredNodalDofMap
    )
    pde = LaplacianWeakForm(dim=PHYSICAL_DIM, f=f_rhs_classical)
    model = AbstractPhysicalWeakModel.from_weak_form(pde, dirichlet=zero_bc)
    assembler = EllipticFEscheme(model, variables)
    return EllipticFEscheme.solve(assembler, max_iter=1)


def make_dg(n_cells, poly_order):
    """Build and solve a DG assembler for the Laplacian on the disk."""
    global POLY_ORDER
    POLY_ORDER = poly_order
    mesh = load_disk_mesh(n_cells)
    basis = AnalyticBasis(
        nb_basis=(poly_order + 1) ** PHYSICAL_DIM,
        out_dim=OUT_DIM,
        mesh=mesh,
        local_basis=taylor,
        local_basis_by_logical=taylor_by_logical,
        basis_type="scalar",
    )
    variables = VariablesDG(basis=basis, nb_variables=OUT_DIM)
    pde = LaplacianWeakForm(dim=PHYSICAL_DIM, f=f_rhs_classical)
    model = AbstractPhysicalWeakModel.from_weak_form(pde, dirichlet=zero_bc)
    sigma = poly_order * (poly_order + 1) * PHYSICAL_DIM
    h = 2.0 / n_cells
    assembler = EllipticDGscheme(model, variables, SIPGFlux(sigma=sigma, h=h))
    return EllipticDGscheme.solve(assembler, max_iter=1)

### The neural method: a PINN

The PINN is built exactly as in the previous tutorials: a domain, a sampler,
a physical model, an approximation space (a multi-layer perceptron), and a
projector that trains the network. Here the Dirichlet condition is enforced
*strongly*: the network output is multiplied by $1 - r^2$ through a
post-processing, so the approximation vanishes on the circle $r = 1$ by
construction, and no boundary collocation points are needed.


In [ ]:
domain_x = Disk2D([0.0, 0.0], 1.0, is_main_domain=True)
domain_mu = [(1.0, 2.0)]
sampler = TensorizedSampler(
    [DomainSampler(domain_x), UniformParametricSampler(domain_mu)], bc=False
)

model_pinn = ParametricLaplacianDirichletND(domain_x, f_rhs, bc="strong")


def post_processing(approx: jnp.ndarray, xy: jnp.ndarray, mu: jnp.ndarray) -> jnp.ndarray:
    r"""Post-processing enforcing the Dirichlet condition strongly.

    We write v_theta = A \circ u_theta with A(x, y) = 1 - r^2, which vanishes
    on the boundary of the unit disk, and u_theta is the network output.
    """
    r2 = xy[0] ** 2 + xy[1] ** 2
    return approx * (1.0 - r2)


def make_pinn(hidden_sizes, n_epochs, n_colloc, n_bc_colloc, seed=0,
              matrix_regularization=1e-8, adaptive_matrix_regularization=True):
    """Build and train a PINN for the parametric Laplacian.

    The Gram matrix of the ENG optimizer is regularized with
    ``matrix_regularization``, and the step size is chosen by a strong-Wolfe
    line search.
    """
    key = jax.random.PRNGKey(seed)
    nn = MLP(
        in_size=3,  # x, y, mu
        out_size=1,
        hidden_sizes=hidden_sizes,
        key=key,
    )
    space = ApproximationSpace(
        {"x": 2},
        [(nn, "scalar", None)],
        model_type="x_mu",
        post_processing=post_processing,  # strong Dirichlet BC: multiply by 1 - r^2
    )
    pinn = Projector(
        model_pinn,
        space,
        sampler,
        adaptive_matrix_regularization=adaptive_matrix_regularization,
        matrix_regularization=matrix_regularization,
        linesearch="strong-wolfe",
    )
    start = timeit.default_timer()
    key, pinn = pinn.project(key, space, n_epochs, n_colloc, n_bc_colloc)
    elapsed = timeit.default_timer() - start
    return pinn, elapsed

### Comparing DoF, error and time

We sweep the classical methods over a range of mesh resolutions and
polynomial orders, and the PINN over a range of network sizes and training
budgets. For each configuration we record the number of degrees of freedom,
the relative $L^2$ error and the computation time.


In [ ]:
# --- classical sweep ---
fem_results = []
dg_results = []

N_CELLS_SWEEP = TODO("mesh resolutions (number of cells per direction) to sweep over, from coarse to fine")
POLY_ORDER_SWEEP = TODO("polynomial orders to sweep over, from low to high")
for n_cells in N_CELLS_SWEEP:
    for poly_order in POLY_ORDER_SWEEP:
        t0 = timeit.default_timer()
        asm = make_fem(n_cells, poly_order)
        t_fem = timeit.default_timer() - t0
        ndof = asm.variables.dofsl.size
        err = float(l2_error(asm, u_exact, relative=True))
        fem_results.append((poly_order, ndof, err, t_fem))
        print(f"FEM  {n_cells}x{n_cells} Q{poly_order}: ndof={ndof:>6}  "
              f"err={err:.3e}  time={t_fem:.3f}s")

        t0 = timeit.default_timer()
        asm = make_dg(n_cells, poly_order)
        t_dg = timeit.default_timer() - t0
        ndof = asm.variables.dofsl.size
        err = float(l2_error(asm, u_exact, relative=True))
        dg_results.append((poly_order, ndof, err, t_dg))
        print(f"DG   {n_cells}x{n_cells} Q{poly_order}: ndof={ndof:>6}  "
              f"err={err:.3e}  time={t_dg:.3f}s")

# --- neural sweep ---
pinn_results = []
PINN_HIDDEN_SWEEP = TODO("network sizes to sweep over, each a list of hidden-layer widths, from small to large")
PINN_EPOCHS_SWEEP = TODO("numbers of training epochs to sweep over, from few to many")
PINN_N_COLLOC = TODO("number of interior collocation points used to train the PINN")
PINN_N_BC_COLLOC = TODO("number of boundary collocation points used to train the PINN")
for hidden in PINN_HIDDEN_SWEEP:
    for n_epochs in PINN_EPOCHS_SWEEP:
        pinn, elapsed = make_pinn(hidden, n_epochs, n_colloc=PINN_N_COLLOC, n_bc_colloc=PINN_N_BC_COLLOC)
        ndof = pinn.space.ndof
        # compute the relative L2 error on random points of the disk
        key = jax.random.PRNGKey(42)
        n_pts = 20000
        r = jnp.sqrt(jax.random.uniform(key, (n_pts, 1)))  # uniform in the disk
        theta = jax.random.uniform(key, (n_pts, 1), minval=0.0, maxval=2.0 * jnp.pi)
        xy = jnp.concatenate([r * jnp.cos(theta), r * jnp.sin(theta)], axis=1)
        mu = jax.random.uniform(key, (n_pts, 1), minval=1.0, maxval=2.0)
        u_num = pinn.evaluate(xy, mu)
        u_ex = analytic_solution(xy, mu)
        err = float(jnp.linalg.norm(u_num - u_ex) / jnp.linalg.norm(u_ex))
        pinn_results.append((n_epochs, ndof, err, elapsed))

### Plotting the results

We plot the error against the number of degrees of freedom, and the
computation time against the error, for each method.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# --- error vs DoF ---
# classical results are linked per polynomial order, not per method
ax = axes[0]
for method, results, marker in (("FEM", fem_results, "o"), ("DG", dg_results, "s")):
    for poly_order in POLY_ORDER_SWEEP:
        rs = [r for r in results if r[0] == poly_order]
        ax.loglog([r[1] for r in rs], [r[2] for r in rs],
                  marker + "-", label=f"{method} Q{poly_order}")
for n_epochs in PINN_EPOCHS_SWEEP:
    rs = [r for r in pinn_results if r[0] == n_epochs]
    ax.loglog([r[1] for r in rs], [r[2] for r in rs],
              "^-", label=f"PINN {n_epochs} epochs")
ax.set_xlabel("Number of degrees of freedom")
ax.set_ylabel("Relative L2 error")
ax.set_title("Error vs. DoF")
ax.legend()
ax.grid(True, which="both", alpha=0.3)

# --- time vs error ---
ax = axes[1]
for method, results, marker in (("FEM", fem_results, "o"), ("DG", dg_results, "s")):
    for poly_order in POLY_ORDER_SWEEP:
        rs = [r for r in results if r[0] == poly_order]
        ax.loglog([r[2] for r in rs], [r[3] for r in rs],
                  marker + "-", label=f"{method} Q{poly_order}")
for n_epochs in PINN_EPOCHS_SWEEP:
    rs = [r for r in pinn_results if r[0] == n_epochs]
    ax.loglog([r[2] for r in rs], [r[3] for r in rs],
              "^-", label=f"PINN {n_epochs} epochs")
ax.set_xlabel("Relative L2 error")
ax.set_ylabel("Computation time (s)")
ax.set_title("Time vs. error")
ax.legend()
ax.grid(True, which="both", alpha=0.3)

plt.tight_layout()
plt.show()

## 2/ A 2D rotating transport equation: DG, PINN, Discrete PINN and Neural SL

We now consider the time-dependent advection equation

$$\partial_t u + \mathbf{a} \cdot \nabla u = 0 \quad \text{in } \Omega \times (0, T)$$

with the rigid rotation velocity field $\mathbf{a}(x) = (-2\pi x_2, 2\pi x_1)$
on the unit disk $\Omega = \{(x,y) : x^2 + y^2 \le 1\}$, and initial condition

$$u_0(x,y) = \exp\!\big(-150\,((x - 0.3)^2 + y^2)\big).$$

The exact solution is the initial bump rotated rigidly by the angle $2\pi t$:

$$u(x,y,t) = u_0\big(R(-2\pi t)\, (x,y)\big),$$

where $R(\theta)$ is the rotation matrix.

We compare four methods: a classical DG method (time-stepped with a
Runge-Kutta scheme), a PINN (the standard physics-informed neural network),
a Discrete PINN (Runge-Kutta in time, a PINN at each stage) and a Neural
Semi-Lagrangian method (the network is fitted to the solution at the foot of
the characteristic at each time step).


### Imports for the transport problem


In [ ]:
from scimba_jax.domains.meshless_domains.domains_2d import Disk2D
from scimba_jax.nonlinear_approximation.numerical_solvers.discrete_pinns import (
    DiscretePINN,
)
from scimba_jax.nonlinear_approximation.numerical_solvers.neural_semi_lagrangian import (
    NeuralSemiLagrangian,
)
from scimba_jax.physical_models.temporal_pde.advection import (
    AdvectionND,
    SteadyAdvectionND,
)
from scimba_jax.time_discrete.butcher_tableau import build_rk2_tableau

### The problem data


In [ ]:
DISK_CENTER = (0.0, 0.0)
DISK_RADIUS = 1.0
DOM_X = Disk2D(DISK_CENTER, DISK_RADIUS, is_main_domain=True)
DOM_T = (0.0, 0.5)
NT = TODO("number of time steps over the interval [0, T] (enough to resolve one full rotation)")
DT = (DOM_T[1] - DOM_T[0]) / NT


def advection_field(x: jnp.ndarray) -> jnp.ndarray:
    """Rigid rotation velocity field a(x) = (-2*pi*x2, 2*pi*x1)."""
    x1, x2 = x[..., 0:1], x[..., 1:2]
    return jnp.stack([-2.0 * jnp.pi * x2, 2.0 * jnp.pi * x1], axis=-1)


def exact_sol(t: jnp.ndarray, x: jnp.ndarray) -> jnp.ndarray:
    """Exact solution: the initial bump rotated by the angle 2*pi*t."""
    x = jnp.atleast_2d(x)
    t = jnp.atleast_2d(t)
    angle = -2.0 * jnp.pi * t[:, 0]
    x1, x2 = x[:, 0], x[:, 1]
    x1_0 = jnp.cos(angle) * x1 - jnp.sin(angle) * x2
    x2_0 = jnp.sin(angle) * x1 + jnp.cos(angle) * x2
    return jnp.exp(-150.0 * ((x1_0 - 0.3) ** 2 + x2_0**2)).reshape(-1, 1)


def f_init(x: jnp.ndarray) -> jnp.ndarray:
    """Initial condition."""
    t = jnp.zeros_like(x)
    return exact_sol(t, x)


def exact_characteristic_foot(t: jnp.ndarray, x: jnp.ndarray, dt: float) -> jnp.ndarray:
    """Foot of the characteristic: rotate x backwards by the angle 2*pi*dt."""
    angle = -2.0 * jnp.pi * dt
    x1, x2 = x[0], x[1]
    x1_foot = jnp.cos(angle) * x1 - jnp.sin(angle) * x2
    x2_foot = jnp.sin(angle) * x1 + jnp.cos(angle) * x2
    return jnp.stack([x1_foot, x2_foot])

### The neural methods

We build the three neural methods, a PINN, a Discrete PINN and a Neural
Semi-Lagrangian method. Each is trained on the same domain and with the same
number of collocation points, so that the comparison is fair.


In [ ]:
N_COLLOC = 2000
N_BC_COLLOC = 1000
N_EPOCHS_INIT = 200
N_EPOCHS = TODO("number of training epochs per time step for the transport neural methods")
TRANSPORT_HIDDEN = TODO("network size (list of hidden-layer widths) shared by the three transport neural methods")

sampler_t = TensorizedSampler([DomainSampler(DOM_X)], model_type="x")


# --- PINN ---
def make_pinn_transport(hidden_sizes, n_epochs, seed=0):
    """Build and train a PINN for the rotating advection equation."""
    key = jax.random.PRNGKey(seed)
    nn = MLP(in_size=3, out_size=1, hidden_sizes=hidden_sizes, key=key)
    space = ApproximationSpace({"x": 2}, [(nn, "scalar", None)], model_type="t_x")
    model = AdvectionND(
        main_domain=DOM_X,
        time_domain=DOM_T,
        velocity=advection_field,
        bc="weak",
        ic="weak",
        f_ic_rhs=f_init,
    )
    pinn = Projector(model, space, sampler_t)
    start = timeit.default_timer()
    key, pinn = pinn.project(key, space, n_epochs, N_COLLOC, N_BC_COLLOC)
    elapsed = timeit.default_timer() - start
    return pinn, elapsed


# --- Discrete PINN ---
def make_discrete_pinn(hidden_sizes, n_epochs, seed=0):
    """Build and train a Discrete PINN for the rotating advection equation."""
    key = jax.random.PRNGKey(seed)
    nn = MLP(in_size=3, out_size=1, hidden_sizes=hidden_sizes, key=key)
    space = ApproximationSpace({"x": 2}, [(nn, "scalar", None)], model_type="x")

    pde = SteadyAdvectionND(DOM_X, DOM_T, advection_field)
    dpinn = DiscretePINN(
        DOM_X, DOM_T, sampler_t, out_size=1, nt=NT,
        butcher_tableau=build_rk2_tableau(),
        explicit_pde=pde,
        implicit_pde=None,
        exact_solution=exact_sol,
    )
    start = timeit.default_timer()
    key, dpinn = dpinn.initialize(key, space, f_init, N_EPOCHS_INIT, N_COLLOC,
                                  n_bc_colloc=N_BC_COLLOC)
    space = dpinn.space
    key, dpinn = dpinn.solve(key, space, n_epochs, N_COLLOC, n_bc_colloc=N_BC_COLLOC)
    elapsed = timeit.default_timer() - start
    return dpinn, elapsed


# --- Neural Semi-Lagrangian ---
def make_neural_sl(hidden_sizes, n_epochs, seed=0):
    """Build and train a Neural Semi-Lagrangian method for the rotating advection."""
    key = jax.random.PRNGKey(seed)
    nn = MLP(in_size=2, out_size=1, hidden_sizes=hidden_sizes, key=key)
    space = ApproximationSpace({"x": 2}, [(nn, "scalar", None)], model_type="x")

    nsl = NeuralSemiLagrangian(
        main_domain=DOM_X,
        time_domain=DOM_T,
        sampler=sampler_t,
        dt=DT,
        advection_field=advection_field,
        exact_characteristic_foot=exact_characteristic_foot,
        periodic=False,
        out_size=1,
        model_type="x",
        exact_solution=exact_sol,
    )
    start = timeit.default_timer()
    key, nsl = nsl.initialize(key, space, f_init, N_EPOCHS_INIT, N_COLLOC)
    space = nsl.space
    key, nsl = nsl.solve(key, space, n_epochs, N_COLLOC)
    elapsed = timeit.default_timer() - start

### Comparing DoF, error and time

We train each method with a fixed network size and record the number of
degrees of freedom, the final relative $L^2$ error and the computation time.


In [ ]:
transport_results = []

# --- PINN ---
pinn, elapsed = make_pinn_transport(TRANSPORT_HIDDEN, N_EPOCHS)
ndof = pinn.space.ndof
key = jax.random.PRNGKey(42)
n_pts = 20000
xy = jax.random.uniform(key, (n_pts, 2), minval=-1.0, maxval=1.0)
# keep only points inside the disk
mask = xy[:, 0] ** 2 + xy[:, 1] ** 2 <= 1.0
xy = xy[mask]
t_final = jnp.ones((xy.shape[0], 1)) * DOM_T[1]
u_num = pinn.evaluate(t_final, xy)
u_ex = exact_sol(t_final, xy)
err = float(jnp.linalg.norm(u_num - u_ex) / jnp.linalg.norm(u_ex))
transport_results.append(("PINN", ndof, err, elapsed))
print(f"PINN: ndof={ndof:>6}  err={err:.3e}  time={elapsed:.3f}s")

# --- Discrete PINN ---
dpinn, elapsed = make_discrete_pinn(TRANSPORT_HIDDEN, N_EPOCHS)
ndof = dpinn.space.ndof
errors = dpinn.errors_over_time
err = float(errors[-1, 0])
transport_results.append(("Discrete PINN", ndof, err, elapsed))
print(f"Discrete PINN: ndof={ndof:>6}  err={err:.3e}  time={elapsed:.3f}s")

# --- Neural Semi-Lagrangian ---
nsl, elapsed = make_neural_sl(TRANSPORT_HIDDEN, N_EPOCHS)
ndof = nsl.space.ndof
errors = nsl.errors_over_time
err = float(errors[-1, 0])
transport_results.append(("Neural SL", ndof, err, elapsed))
print(f"Neural SL: ndof={ndof:>6}  err={err:.3e}  time={elapsed:.3f}s")

### Plotting the results

We plot the final error and the computation time of each method as a bar chart.


In [ ]:
methods = [r[0] for r in transport_results]
errors = [r[2] for r in transport_results]
times = [r[3] for r in transport_results]

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

ax = axes[0]
ax.bar(methods, errors, color="steelblue")
ax.set_ylabel("Relative L2 error (final time)")
ax.set_title("Final error")
ax.tick_params(axis="x", rotation=15)

ax = axes[1]
ax.bar(methods, times, color="coral")
ax.set_ylabel("Computation time (s)")
ax.set_title("Computation time")
ax.tick_params(axis="x", rotation=15)

plt.tight_layout()
plt.show()

### Takeaways

Classical methods (FEM, DG) solve a *linear system* whose size is fixed by the
mesh. Their cost scales with the number of degrees of freedom, and the error
decreases at a predictable rate as the mesh is refined. Neural methods (PINN,
Discrete PINN, Neural SL) *optimize* the weights of a network. Their cost is
governed by the number of training epochs and collocation points, not by the
mesh. The error decreases with the number of epochs, but the convergence is
slower and less predictable than for classical methods.

The two families of methods are therefore not directly comparable. A fair
comparison requires fixing a target accuracy and asking how much work each
method needs to reach it. In many cases, classical methods are faster and more
accurate for problems that can be discretized on a mesh, while neural methods
are more flexible for problems with complex geometries, high dimensionality,
or parameters that are difficult to handle with a mesh.

Don't hesitate to play with the hyperparameters and the mesh resolutions to
explore the trade-offs between the two families of methods!